# Ejemplo de Agente aplicando los principios MATE
Este notebook construye un **agente de recordatorios** 

- **M**odel efficiency → una tool "rápida" (`fast_llm`) para una tarea simple (normalizar fecha)
- **A**ction specificity → una sola tool con propósito claro (`create_reminder_safely`), no una genérica tipo `update_reminder`
- **T**oken efficiency → prompts cortos y enfocados, sin contexto de más
- **E**nvironmental safety → la acción es reversible: guardamos cómo deshacerla y damos al agente una tool `undo_last_action`

In [1]:
%run "4.1_frameworkAgent.ipynb"

18:39:05 - LiteLLM:WARNING: get_model_cost_map.py:454 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [Errno -3] Temporary failure in name resolution. Falling back to local backup.


In [2]:
# Base de datos simulada
reminders_db = {}

In [3]:
# Tool principal: crear un recordatorio de forma segura
@register_tool(tags=["reminders", "safe"])
def create_reminder_safely(action_context: ActionContext, title: str, date: str, note: str = "") -> dict:
    """
    Crea un recordatorio de forma segura: valida los datos, normaliza la fecha,
    lo guarda, y registra cómo deshacerlo si hace falta.

    Args:
        title: Título del recordatorio (obligatorio, no puede estar vacío)
        date: Fecha en cualquier formato de texto libre (ej. "mañana", "10 de marzo")
        note: Nota opcional adicional

    Returns:
        Un diccionario con el id del recordatorio creado y sus datos normalizados
    """
    # --- Action specificity: validación incluida en la propia tool ---
    if not title.strip():
        raise ValueError("El recordatorio necesita un título")

    # --- Model efficiency: modelo "rápido" para una tarea simple ---
    # En un caso real, fast_llm apuntaría a un modelo más pequeño/barato
    # (ej. un modelo tipo 8B) y powerful_llm a uno más capaz para tareas complejas.
    fast_llm = action_context.get("fast_llm") or action_context.get("llm")

    # --- Token efficiency: prompt corto, una sola instrucción clara ---
    normalized_date = fast_llm(Prompt(messages=[
        {"role": "system", "content": "Devuelve solo la fecha en formato YYYY-MM-DD, sin texto extra."},
        {"role": "user", "content": date}
    ])).strip()

    reminder_id = str(uuid.uuid4())[:8]
    reminders_db[reminder_id] = {"title": title, "date": normalized_date, "note": note}

    # --- Environmental safety: registramos cómo deshacer esta acción ---
    action_context.set("last_action", {
        "type": "create_reminder",
        "reminder_id": reminder_id
    })

    return {"reminder_id": reminder_id, "title": title, "date": normalized_date, "note": note}

In [4]:
## Tool de seguridad: deshacer la última acción
@register_tool(tags=["reminders", "safe"])
def undo_last_action(action_context: ActionContext) -> dict:
    """
    Deshace la última acción reversible que registró el agente.
    Úsala si el usuario pide cancelar o deshacer lo último que se hizo.

    Returns:
        Confirmación de qué se deshizo, o un aviso si no había nada que deshacer
    """
    last = action_context.get("last_action")
    if not last:
        return {"status": "nothing_to_undo"}

    if last["type"] == "create_reminder":
        removed = reminders_db.pop(last["reminder_id"], None)
        action_context.set("last_action", None)
        return {"status": "undone", "removed_reminder": removed}

    return {"status": "unknown_action_type"}

In [5]:
# ensamblar agente
def create_reminder_agent():
    action_registry = PythonActionRegistry(tags=["reminders", "system"])
    environment = PythonEnvironment()

    goals = [
        Goal(
            name="Persona",
            description="Eres un asistente de recordatorios cuidadoso y directo."
        ),
        Goal(
            name="Gestionar recordatorios",
            description="""
            Cuando el usuario pida crear un recordatorio, usa create_reminder_safely.
            Si el usuario pide cancelar, deshacer, o dice que se equivocó,
            usa undo_last_action.
            Confirma siempre con un mensaje breve y termina la tarea.
            """
        )
    ]

    return Agent(
        goals=goals,
        agent_language=AgentFunctionCallingActionLanguage(),
        action_registry=action_registry,
        generate_response=generate_response,
        environment=environment
    )


agent = create_reminder_agent()

# Model efficiency: aquí es donde "conectarías" un modelo distinto y más barato
# para tareas simples. Como el curso solo tiene un modelo configurado, usamos
# el mismo generate_response para ilustrar el patrón.
agent.action_context.set("fast_llm", generate_response)
agent.action_context.set("powerful_llm", generate_response)

In [7]:
# crear un recordatorio
memory = agent.run("Crea un recordatorio para mañana: pagar la colegiatura")
print(reminders_db)

Agent thinking...
Agent Decision: {"tool": "create_reminder_safely", "args": {"date": "ma\u00f1ana", "note": "", "title": "pagar la colegiatura"}}
Action Result: {'tool_executed': True, 'result': {'reminder_id': '3444919f', 'title': 'pagar la colegiatura', 'date': '2026-09-02', 'note': ''}, 'timestamp': '2026-08-31T18:39:35-0600'}
Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Recordatorio creado: **pagar la colegiatura** para el **2026\u201109\u201102**."}}
Action Result: {'tool_executed': True, 'result': 'Recordatorio creado: **pagar la colegiatura** para el **2026‑09‑02**.\nTerminando...', 'timestamp': '2026-08-31T18:39:35-0600'}
{'3444919f': {'title': 'pagar la colegiatura', 'date': '2026-09-02', 'note': ''}}


In [8]:
# deshacer el recordatorio
memory = agent.run("Mejor cancela ese recordatorio, me equivoqué", memory=memory)
print(reminders_db)

Agent thinking...
Agent Decision: {"tool": "undo_last_action", "args": {}}
Action Result: {'tool_executed': True, 'result': {'status': 'undone', 'removed_reminder': {'title': 'pagar la colegiatura', 'date': '2026-09-02', 'note': ''}}, 'timestamp': '2026-08-31T18:39:45-0600'}
Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "Recordatorio\u00a0\u00a1\u00a1\u00a1\u00a0\u00a0\u00a0\u00a0\u00a0\u00a0\u2026\n\nRecordatorio cancelado correctamente."}}
Action Result: {'tool_executed': True, 'result': 'Recordatorio\xa0¡¡¡\xa0\xa0\xa0\xa0\xa0\xa0…\n\nRecordatorio cancelado correctamente.\nTerminando...', 'timestamp': '2026-08-31T18:39:46-0600'}
{}
